# QLoRA Adapter Training

Train per-artist LoRA/DoRA adapters on Gemma 4 E4B.

In [1]:
from huggingface_hub import login
login()

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/gemma-4-E4B",
    local_dir="./models/gemma-4-E4B",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/home/aliozkaya/uni/467/term_project/src/models/gemma-4-E4B'

In [ ]:
from peft import prepare_model_for_kbit_training

from generation.model import load_base_model
from generation.data import load_train_df
from generation.train import train_adapter
from generation.style_loss import build_style_weights, top_tokens

model, tokenizer = load_base_model()
model = prepare_model_for_kbit_training(model)

train_df = load_train_df()

## Main adapters (LoRA)

In [ ]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=False)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=False)

## Ablation: DoRA (same artist)

In [ ]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=True)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=True)

## Ablation: Rank

In [ ]:
train_adapter(model, tokenizer, train_df, "Gojira", r=4, use_dora=False)
train_adapter(model, tokenizer, train_df, "Gojira", r=16, use_dora=False)

## Remaining artists (LoRA + DoRA r=8)

Completes the lineup so the attribution table and perplexity diagonal cover all 5 artists, and the LoRA-vs-DoRA ablation generalizes beyond Gojira/Tool.

In [ ]:
for artist in ["Death", "Meshuggah", "Opeth"]:
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=False)
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=True)

## Experiment: Style-Weighted Loss

Up-weight artist-distinctive tokens (token-level TF-IDF) in the cross-entropy so
the adapter learns artist vocabulary, not just genre. Building/inspecting the
weights is CPU-only; only the `train_adapter` call below needs the GPU. Trains to
`*_sw` adapters so they can be A/B'd against the plain ones on attribution.

In [ ]:
style_w = build_style_weights("Gojira", train_df, tokenizer, text_col="clean")
top_tokens(style_w, tokenizer, n=30)

In [ ]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=False, style_weights=style_w)